# 🍪 SQL JOIN 탐험대: 학교 간식 가게의 비밀
### 초등학생도 이해하는 데이터베이스 결합 — 90분 실습

> **오늘의 이야기:** 학교 간식 가게에는 **회원 명단**과 **주문 장부**가 따로 있어요. 두 장부를 연결해 누가 어떤 간식을 샀는지 알아봅시다!

**학습 목표** ① 테이블·행·열·ID 이해하기 ② INNER / LEFT / RIGHT / FULL OUTER JOIN 비교하기 ③ 주문하지 않은 회원 찾기 ④ 회원별 주문 횟수와 총액 계산하기

**사용 방법:** 구글 코랩(Colab)에 이 파일을 올리고, 코드 셀의 ▶ 버튼을 **위에서 아래로** 누르세요. SQL은 `질문 문장`이라고 생각하면 됩니다. Python은 SQL이 실행되도록 돕는 도구입니다.

**90분 안내:** 이야기·데이터 관찰 15분 → INNER 15분 → LEFT 15분 → RIGHT/FULL 15분 → 실습 미션 20분 → 퀴즈 10분

## 1. 테이블은 무엇일까요? (15분)

- **테이블(Table)** = 이름표가 붙은 표 한 장
- **열(Column)** = 어떤 정보를 적는지 알려주는 제목(이름, 상품, 가격)
- **행(Row)** = 한 사람 또는 한 번의 주문을 적은 한 줄
- **기본키(PK)** = 각 회원에게 하나씩 주는 **겹치지 않는 번호표** `id`
- **외래키(FK)** = 주문 장부에 적힌 **회원 번호표** `user_id`. 이 번호로 두 장부를 연결합니다.
- **NULL** = 빈칸, 즉 '값이 적혀 있지 않음'. **숫자 0이나 빈 문자열과 달라요.**

**왜 장부를 나눌까요?** 지민이가 과자를 두 번 사도 이름과 학년은 회원 장부에 딱 한 번만 적으면 됩니다. 주문 장부는 회원번호만 기억하면 되죠.

**연결 규칙**: `Users.id = Orders.user_id` (회원 번호가 같을 때 연결!)

### 🧩 우리 가게의 두 장부

**회원 명단 `Users`**

| id | name | grade |
|---:|---|---:|
| 101 | 지민 | 4 |
| 102 | 민수 | 5 |
| 103 | 서아 | 4 |
| 104 | 준호 | 6 |
| 105 | 하린 | 5 |

**주문 장부 `Orders`**

| order_id | user_id | item | price |
|---:|---:|---|---:|
| 1 | 101 | 초코우유 | 1500 |
| 2 | 101 | 쿠키 | 1000 |
| 3 | 102 | 젤리 | 2000 |
| 4 | 104 | 빵 | 2500 |
| 5 | NULL | 사과주스 | 1800 |

**생각해 봐요:** ① 지민이는 몇 번 샀나요? ② 아직 한 번도 사지 않은 회원은? ③ 회원번호가 없는 사과주스 주문은 누구 것인지 알 수 있을까요?

> 마지막 주문은 **회원으로 등록하지 않은 방문 손님**이 샀다고 가정합니다. 그래서 회원번호가 NULL이에요. 데이터가 잘못된 주문이 아닙니다.

In [ ]:
import sqlite3
import pandas as pd

# 코랩에서 메모리 속 미니 데이터베이스를 만들어요.
# 이 셀을 다시 실행하면 이전 예제를 깨끗하게 초기화합니다.
try:
    conn.close()
except (NameError, Exception):
    pass
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
CREATE TABLE Users (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    grade INTEGER NOT NULL
);
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    item TEXT NOT NULL,
    price INTEGER NOT NULL CHECK (price >= 0),
    FOREIGN KEY (user_id) REFERENCES Users(id)
);
INSERT INTO Users (id, name, grade) VALUES
(101, '지민', 4), (102, '민수', 5), (103, '서아', 4),
(104, '준호', 6), (105, '하린', 5);
INSERT INTO Orders (order_id, user_id, item, price) VALUES
(1, 101, '초코우유', 1500), (2, 101, '쿠키', 1000),
(3, 102, '젤리', 2000), (4, 104, '빵', 2500),
(5, NULL, '사과주스', 1800);
""")

def show(sql):
    """SQL 문장을 실행하고 표로 보여 주는 도우미"""
    return pd.read_sql_query(sql, conn)

print('✅ 가게 장부 준비 완료! SQLite 버전:', sqlite3.sqlite_version)
print('회원 5명, 주문 5건이 들어 있습니다.')

In [ ]:
print('👦 회원 명단')
display(show('SELECT * FROM Users ORDER BY id'))
print('🍬 주문 장부')
display(show('SELECT * FROM Orders ORDER BY order_id'))

## 2. INNER JOIN — 양쪽에서 번호가 같은 친구만! (15분)

**비유:** 회원 명단에서 이름표를 찾고, 주문 장부에서 같은 번호를 찾아 **짝을 만들어 줍니다**. 짝을 못 찾으면 결과에 나오지 않아요.

한 회원이 여러 번 주문했다면 결과에도 **여러 줄**이 나와요. (한 명 = 한 줄이 아닐 수 있음!)

읽는 순서: `SELECT` 보여 줄 열 → `FROM` 첫 장부 → `INNER JOIN` 둘째 장부 → `ON` 짝 맞출 기준.

In [ ]:
display(show("""SELECT U.id AS 회원번호, U.name AS 이름,
       O.item AS 간식, O.price AS 가격
FROM Users AS U
INNER JOIN Orders AS O
  ON U.id = O.user_id
ORDER BY O.order_id;"""))
print("✅ 예상: 4줄(지민 2줄, 민수 1줄, 준호 1줄)")

**확인:** 서아와 하린은 주문이 없어 나오지 않고, 방문 손님의 사과주스는 회원 이름과 연결되지 않아 나오지 않습니다.

**미니 질문:** 지민이가 왜 두 줄일까요? → 주문이 두 건이기 때문입니다.

## 3. LEFT JOIN — 왼쪽 회원은 모두 데려와요! (15분)

**비유:** 선생님이 반 전체 학생 명단을 들고 출석을 부릅니다. 구매한 내역이 없어도 **회원 명단에 있는 사람은 모두 보입니다.** 주문이 없으면 오른쪽 주문 칸이 `NULL`입니다.

여기서 왼쪽은 `FROM Users` 테이블이에요.

In [ ]:
display(show("""SELECT U.id AS 회원번호, U.name AS 이름,
       O.item AS 간식, O.price AS 가격
FROM Users AS U
LEFT JOIN Orders AS O
  ON U.id = O.user_id
ORDER BY U.id, O.order_id;"""))
print("✅ 예상: 6줄(지민 2줄 + 다른 회원 4줄)")

### 🔎 특별 미션: 간식을 한 번도 사지 않은 회원 찾기

`WHERE O.order_id IS NULL`의 뜻: **짝을 맞춰 봤는데 주문 번호가 비어 있는 회원만 보여 줘!**

`IS NULL`은 빈칸 검사이고, `= NULL`로 쓰지 않습니다.

In [ ]:
display(show("""SELECT U.name AS 주문_없는_회원
FROM Users AS U
LEFT JOIN Orders AS O
  ON U.id = O.user_id
WHERE O.order_id IS NULL
ORDER BY U.id;"""))
print("✅ 정답: 서아, 하린")

## 4. RIGHT JOIN — 오른쪽 주문은 모두 데려와요! (7분)

이번에는 **오른쪽에 있는 주문 장부**를 모두 보여 줍니다. 회원 번호가 없는 방문 손님의 주문도 빠지지 않습니다. 이름 칸은 `NULL`이에요.

In [ ]:
display(show("""SELECT U.name AS 이름, O.order_id AS 주문번호,
       O.item AS 간식
FROM Users AS U
RIGHT JOIN Orders AS O
  ON U.id = O.user_id
ORDER BY O.order_id;"""))
print("✅ 예상: 5줄(방문 손님의 사과주스 포함)")

## 5. FULL OUTER JOIN — 양쪽 장부를 하나도 빠뜨리지 않아요! (8분)

회원 명단에만 있는 친구도, 주문 장부에만 있는 방문 손님의 주문도 **전부** 보여 줍니다. 따라서 `NULL`은 왼쪽이나 오른쪽 열에서 나올 수 있어요.

> 참고: 이 실습은 SQLite 3.39.0 이상에서 RIGHT/FULL JOIN을 지원합니다. 코랩에서 버전을 위의 첫 코드 셀로 확인할 수 있어요.

In [ ]:
display(show("""SELECT U.name AS 이름, O.order_id AS 주문번호,
       O.item AS 간식
FROM Users AS U
FULL OUTER JOIN Orders AS O
  ON U.id = O.user_id
ORDER BY U.id IS NULL, U.id, O.order_id;"""))
print("✅ 예상: 7줄(연결 4 + 주문 없는 회원 2 + 방문 손님 1)")

### 📌 한눈에 비교

| JOIN | 누구를 꼭 남기나요? | 이 예제 결과 |
|---|---|---:|
| `INNER JOIN` | 양쪽에 같은 번호가 있는 주문 | 4줄 |
| `LEFT JOIN` | 왼쪽 회원 모두 | 6줄 |
| `RIGHT JOIN` | 오른쪽 주문 모두 | 5줄 |
| `FULL OUTER JOIN` | 양쪽에 있는 모든 행 | 7줄 |

**중요:** JOIN 결과의 줄 수는 **서로 짝이 몇 개 생기는지**에 따라 달라집니다. 단순히 두 표의 사람 수를 더하는 것은 아닙니다.

## 6. 회원별 주문 통계 — 숫자도 셀 수 있어요 (10분)

- `GROUP BY` = 같은 회원끼리 모으기
- `COUNT(O.order_id)` = 주문번호가 **있는 행만** 세기 (주문이 없으면 0)
- `COALESCE(SUM(O.price), 0)` = 가격을 모두 더하되, 합계가 `NULL`이라면 0으로 보여 주기

주의: `COUNT(*)`를 쓰면 주문이 없는 회원도 LEFT JOIN 결과에서 한 줄을 차지하므로 **1건으로 잘못 셀 수 있어요.**

In [ ]:
display(show("""SELECT U.name AS 이름,
       COUNT(O.order_id) AS 주문횟수,
       COALESCE(SUM(O.price), 0) AS 총결제금액
FROM Users AS U
LEFT JOIN Orders AS O
  ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY U.id;"""))
print("✅ 지민 2회/2500원, 민수 1회/2000원, 서아 0회/0원, 준호 1회/2500원, 하린 0회/0원")

## 7. 내 손으로 푸는 3단계 미션 (20분)

아래 셀에 적힌 `TODO`를 바꿔 쿼리를 실행하세요. **막혀도 괜찮아요.** 위에서 본 문장을 복사해서 조금만 고쳐도 됩니다.

**미션 1 (기본):** 회원 중 **젤리**를 산 사람의 이름과 간식명을 출력하세요. 힌트: `INNER JOIN` + `WHERE O.item = '젤리'`

**미션 2 (기본):** **주문하지 않은 친구 이름**을 찾으세요. 힌트: `LEFT JOIN` + `WHERE ... IS NULL`

**미션 3 (도전):** 회원별 주문 횟수와 결제 합계를 구하세요. 힌트: `LEFT JOIN`, `COUNT`, `SUM`, `GROUP BY`

In [ ]:
# ✏️ 미션 1: TODO 부분을 고쳐 보세요.
q1 = """
SELECT U.name AS 이름, O.item AS 간식
FROM Users U
INNER JOIN Orders O ON U.id = O.user_id
WHERE O.item = '젤리';
"""
display(show(q1))

In [ ]:
# ✏️ 미션 2: 이번에는 문장을 읽고 조건을 설명해 보세요.
q2 = """
SELECT U.name AS 이름
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
WHERE O.order_id IS NULL;
"""
display(show(q2))

In [ ]:
# ✏️ 미션 3: 각 함수가 하는 일을 친구에게 말해 보세요.
q3 = """
SELECT U.name AS 이름,
       COUNT(O.order_id) AS 주문횟수,
       COALESCE(SUM(O.price), 0) AS 총금액
FROM Users U
LEFT JOIN Orders O ON U.id = O.user_id
GROUP BY U.id, U.name
ORDER BY U.id;
"""
display(show(q3))

### 🚀 추가 도전 (선택)

1. **4학년 회원만** 골라 주문 내역을 보여 주세요. (`WHERE U.grade = 4`)
2. **가격이 2,000원 이상**인 주문의 회원 이름과 간식을 보여 주세요.
3. **주문을 두 번 이상** 한 회원을 찾으세요. (`GROUP BY` + `HAVING COUNT(O.order_id) >= 2`)
4. 새로운 회원을 추가해 보세요. `INSERT INTO Users VALUES (106, '다온', 6);` 그런 다음 LEFT JOIN 결과를 다시 보세요.

**생각 문제:** 새 회원을 추가하면 INNER JOIN에는 바로 나타날까요? 이유를 설명하세요.

## 8. 형성평가 — 5문제 (10분)

1. 회원 이름과 주문을 연결하는 데 필요한 공통 번호는 무엇인가요? **(A) 학년 (B) 회원번호 (C) 가격**
2. 주문이 있는 회원의 주문만 보고 싶으면? **(A) INNER (B) LEFT (C) FULL**
3. 주문이 없는 회원도 모두 보고 싶으면? **(A) INNER (B) LEFT (C) RIGHT**
4. `NULL`의 뜻은? **(A) 숫자 0 (B) 값이 없음 (C) 빈칸에 반드시 0이 들어감**
5. LEFT JOIN 후 주문 없는 회원을 찾는 조건은? **(A) `O.order_id = NULL` (B) `O.order_id IS NULL` (C) `O.order_id > 0`**

**정답:** 1-B, 2-A, 3-B, 4-B, 5-B

**마지막 한 문장:** `JOIN`은 **두 장부에서 공통 번호를 찾아 정보를 연결하는 방법**이에요!

**다음 시간:** `GROUP BY`, `HAVING`, `COUNT`, `SUM`을 써서 우리 반 인기 간식 순위를 만들어 봅시다.

---
### 선생님용 참고 · 원본 및 검증

- 원본 PDF *SQL JOIN 수업 발표 자료*, 3~11쪽: 테이블 분리, 4종 JOIN, 미구매 회원 검색, 회원별 통계, 코랩, 형성평가를 순서대로 확장했습니다.
- SQLite 공식 문서: https://www.sqlite.org/lang_select.html (LEFT / RIGHT / FULL JOIN 동작, `ON`과 `WHERE` 차이)
- Python `sqlite3` 공식 문서: https://docs.python.org/3/library/sqlite3.html
- SQLite 공식 기능 안내: https://www.sqlite.org/fullsql.html

**수업 보완 주의점:** `INNER JOIN`도 원본 데이터에 `NULL`이 있으면 선택한 열의 `NULL`이 나올 수 있으므로, '항상 NULL이 없다'라고 일반화하지 않습니다. `LEFT JOIN`을 `WHERE O.price >= 2000`으로 필터링하면 주문 없는 회원은 제외될 수 있습니다. RIGHT/FULL JOIN은 SQLite 3.39 이상 환경에서 실행하세요.